# Radar Hipotecario — EDA e Ingeniería de Características
**Proyecto final Diplomado Ciencia de Datos G33 — UNAM FES Acatlán**

Estructura alineada al formato de referencia del curso (Imports → Global variables → Functions → Data Ingestion → Feature Engineering → Visualización).

**Nota de reproducibilidad:** este notebook NO requiere credenciales ni tokens de API. Todos los datos se leen de snapshots públicos versionados en GitHub (`data/snapshots/latest/`), generados por el pipeline de ingesta del repositorio. Esto garantiza que corra igual en cualquier entorno de Google Colab, sin depender de whitelists de IP ni límites de tasa de las APIs originales (Banxico, INEGI).

### Imports

In [1]:
# Para producción
import io
import json
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

/Users/luiscontreras/Documents/radar-hipotecario/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

### Variables globales

In [3]:
# Repositorio público — sin tokens, sin credenciales
GITHUB_USER = 'ContrerasPeninsula'
GITHUB_REPO = 'radar-hipotecario'
GITHUB_BRANCH = 'main'

RAW_BASE = f'https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}'

URL_SERIES_BANXICO = f'{RAW_BASE}/data/snapshots/latest/series_banxico.parquet'
URL_UMA = f'{RAW_BASE}/config/uma.json'
URL_REGLAS_INFONAVIT = f'{RAW_BASE}/config/reglas_infonavit_v2026.json'

### Funciones

In [4]:
def cargar_parquet_github(url: str) -> pd.DataFrame:
    """
    Descarga un archivo Parquet desde una URL pública de GitHub (raw.githubusercontent.com)
    y lo carga como DataFrame, sin depender de fsspec/credenciales.
    """
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return pd.read_parquet(io.BytesIO(r.content))

In [5]:
def cargar_json_github(url: str) -> dict:
    """Descarga y parsea un archivo JSON público de GitHub."""
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

In [6]:
def calcular_variacion_pct(df: pd.DataFrame, columna: str, periodos: int) -> pd.Series:
    """
    Calcula variación porcentual de una columna a `periodos` observaciones de distancia.
    Para series diarias: periodos=252 aprox. 1 año hábil; periodos=21 aprox. 1 mes.
    """
    return df[columna].pct_change(periods=periodos) * 100

In [7]:
def resumen_estadistico(df: pd.DataFrame, columnas: list) -> pd.DataFrame:
    """Resumen descriptivo (media, std, min, max, percentiles) de columnas numéricas."""
    return df[columnas].describe().T

## Data Ingestion

### Descripción detallada de las fuentes de datos

| Fuente | Tipo | Autenticación | Frecuencia de actualización | Limitación conocida |
|---|---|---|---|---|
| Banxico SIE API | API REST | Token gratuito (header `Bmx-Token`) | Diaria (TIIE, tasa objetivo, FIX); mensual (INPC) | Ninguna relevante — fuente oficial estable |
| UMA (INEGI/DOF) | Constante versionada a mano | N/A | Anual (~1 de febrero) | Requiere actualización manual cada año |
| Tabla de tasas Infonavit | PDF oficial descargado | N/A | Sujeta a cambios del instituto, sin calendario fijo | Validada una vez (2026-07); revalidar si Infonavit publica nueva tabla |
| Índice SHF de Precios de la Vivienda | Datos abiertos oficiales (XLSX) | N/A | Trimestral | Precio medio y percentiles se transcriben de la tabla oficial; variación anual se calcula por código desde el archivo de datos abiertos. Cobertura: 32 entidades federativas |
| Inmuebles24 (scraper) | Selenium + BeautifulSoup | N/A | Manual, bajo demanda | **Sesgo hacia anuncios "Destacado"/pagados** — confirmado empíricamente (ver celda de calidad de datos más abajo). Por ese sesgo, no alimenta ninguna decisión de la app ni el K-Means; se conserva solo como pieza exploratoria documentada |


### Series macroeconómicas (Banxico: TIIE, Tasa objetivo, FIX, INPC)
Fuente: [Banxico SIE API](https://www.banxico.org.mx/SieAPIRest/)

In [8]:
df_banxico = cargar_parquet_github(URL_SERIES_BANXICO)
print(f'Filas: {len(df_banxico):,} | Series: {df_banxico["serie"].nunique()} | '
      f'Rango: {df_banxico["fecha"].min().date()} a {df_banxico["fecha"].max().date()}')
df_banxico.head()

Filas: 8,783 | Series: 4 | Rango: 2016-08-07 a 2026-08-05


,fecha,serie,serie_id,valor
0,2016-08-08,fix_usd,SF43718,18.5716
1,2016-08-09,fix_usd,SF43718,18.3842
2,2016-08-10,fix_usd,SF43718,18.3479
3,2016-08-11,fix_usd,SF43718,18.2678
4,2016-08-12,fix_usd,SF43718,18.2455


In [9]:
df_banxico['serie'].value_counts()

serie
tasa_objetivo    3639
fix_usd          2513
tiie_28          2513
inpc_general      118
Name: count, dtype: int64

In [10]:
# Verificación de completitud: observaciones por serie y huecos de fechas
df_banxico.groupby('serie')['fecha'].agg(['min', 'max', 'count'])

,min,max,count
serie,,,
fix_usd,2016-08-08,2026-08-05,2513
inpc_general,2016-09-01,2026-06-01,118
tasa_objetivo,2016-08-07,2026-08-05,3639
tiie_28,2016-08-08,2026-08-05,2513


**Evaluación de calidad — completitud temporal:** las cuatro series cubren el mismo rango histórico (~10 años), el conteo de observaciones es consistente con la periodicidad esperada de cada serie (diaria para TIIE/tasa objetivo/FIX, mensual para INPC). No se detectan series truncadas ni con inicio tardío que sesguen el entrenamiento de Prophet más adelante.

**Evaluación de calidad — sesgo de fuente (oferta inmobiliaria):** el detalle de anuncios scrapeados de Inmuebles24 se probó al inicio del proyecto como fuente de precio de vivienda. La mediana de precios de esa muestra resultó consistentemente por encima de la mediana oficial de SHF para las mismas entidades — evidencia del sesgo hacia anuncios "Destacado"/pagados que muestra primero el portal. Por eso esta fuente se descartó por completo de cualquier cálculo que vea el usuario final: tanto el K-Means (sección de Modelado) como el posicionamiento de mercado de la app usan exclusivamente el índice oficial de SHF, sin ninguna dependencia del scraper.


### UMA (Unidad de Medida y Actualización)
Constante versionada por vigencia — no proviene de una API, se actualiza manualmente cada febrero cuando INEGI/DOF publican el nuevo valor.

In [11]:
uma_data = cargar_json_github(URL_UMA)
df_uma = pd.DataFrame(uma_data['valores'])
df_uma

,vigencia_inicio,vigencia_fin,diario,mensual
0,2024-02-01,2025-01-31,108.5700,3300.5300
1,2025-02-01,2026-01-31,113.1400,3439.4600
2,2026-02-01,2027-01-31,117.3100,3566.2200


### Tabla de tasas Infonavit (motor de reglas, referencia)
No es una serie de tiempo — es la tabla oficial de tasas diferenciadas por nivel salarial (vigente para UMA 2026), usada como insumo del motor de reglas determinista. Se incluye aquí para el EDA porque describe la distribución de tasas que enfrenta cada segmento de usuarios.

In [12]:
reglas_infonavit = cargar_json_github(URL_REGLAS_INFONAVIT)
df_infonavit_tasas = pd.DataFrame(reglas_infonavit['tasas']['tabla_diferenciada_por_uma'])
print(f"Estado de las reglas: {reglas_infonavit['estado']} | Versión: {reglas_infonavit['version']}")
df_infonavit_tasas.head(10)

Estado de las reglas: VALIDADO | Versión: 2026.07-VALIDADO


,uma_min,uma_max,tasa,salario_mensual_referencia
0,0.0000,2.6000,0.0369,9272.1800
1,2.6000,2.7000,0.0388,9628.8000
2,2.7000,2.8000,0.0407,9985.4300
3,2.8000,2.9000,0.0426,10342.0500
4,2.9000,3.0000,0.0445,10698.6700
5,3.0000,3.1000,0.0464,11055.2900
6,3.1000,3.2000,0.0483,11411.9200
7,3.2000,3.3000,0.0502,11768.5400
8,3.3000,3.4000,0.0521,12125.1600
9,3.4000,3.5000,0.0540,12481.7800


## Feature Engineering

### Series macro: pivote a formato ancho y variables derivadas

In [13]:
# Pivote: una columna por serie, indexado por fecha — facilita features cruzados entre series
df_wide = df_banxico.pivot_table(index='fecha', columns='serie', values='valor').sort_index()
df_wide = df_wide.ffill()  # las series no cotizan todos los mismos días; forward-fill conservador
df_wide.tail()

serie,fix_usd,inpc_general,tasa_objetivo,tiie_28
fecha,,,,
2026-08-01,17.3288,145.1310,6.5000,6.7458
2026-08-02,17.3288,145.1310,6.5000,6.7458
2026-08-03,17.3317,145.1310,6.5000,6.7559
2026-08-04,17.2717,145.1310,6.5000,6.8061
2026-08-05,17.2317,145.1310,6.5000,6.7458


In [14]:
# Inflación interanual a partir del INPC (Índice Nacional de Precios al Consumidor)
if 'inpc_general' in df_wide.columns:
    df_wide['inflacion_anual_pct'] = calcular_variacion_pct(df_wide, 'inpc_general', periodos=252)
    df_wide['inflacion_mensual_pct'] = calcular_variacion_pct(df_wide, 'inpc_general', periodos=21)
df_wide[['inpc_general', 'inflacion_mensual_pct', 'inflacion_anual_pct']].tail()

serie,inpc_general,inflacion_mensual_pct,inflacion_anual_pct
fecha,,,
2026-08-01,145.1310,0.0000,1.7428
2026-08-02,145.1310,0.0000,1.7428
2026-08-03,145.1310,0.0000,1.7428
2026-08-04,145.1310,0.0000,1.7428
2026-08-05,145.1310,0.0000,1.7428


In [15]:
# Medias móviles de la TIIE — suavizan el ruido de ajustes discretos de política monetaria
df_wide['tiie_28_ma30'] = df_wide['tiie_28'].rolling(window=30).mean()
df_wide['tiie_28_ma90'] = df_wide['tiie_28'].rolling(window=90).mean()
df_wide[['tiie_28', 'tiie_28_ma30', 'tiie_28_ma90']].tail()

serie,tiie_28,tiie_28_ma30,tiie_28_ma90
fecha,,,
2026-08-01,6.7458,6.7579,6.7718
2026-08-02,6.7458,6.7532,6.7689
2026-08-03,6.7559,6.7532,6.7660
2026-08-04,6.8061,6.7549,6.7625
2026-08-05,6.7458,6.7545,6.7595


In [16]:
# Spread TIIE vs Tasa objetivo — brecha relevante para detectar presión de mercado
df_wide['spread_tiie_objetivo'] = df_wide['tiie_28'] - df_wide['tasa_objetivo']
df_wide['spread_tiie_objetivo'].describe()

count   3639.0000
mean       0.2850
std        0.0923
min       -0.3587
25%        0.2450
50%        0.2680
75%        0.3413
max        0.8280
Name: spread_tiie_objetivo, dtype: float64

In [17]:
# Features de fecha, útiles para segmentar el EDA por periodo
df_wide['anio'] = df_wide.index.year
df_wide['mes'] = df_wide.index.month
df_wide['trimestre'] = df_wide.index.quarter
df_wide[['anio', 'mes', 'trimestre']].tail()

serie,anio,mes,trimestre
fecha,,,
2026-08-01,2026,8,3
2026-08-02,2026,8,3
2026-08-03,2026,8,3
2026-08-04,2026,8,3
2026-08-05,2026,8,3


### Motor de reglas Infonavit: variables descriptivas de la tabla de tasas

In [18]:
# Pendiente de la curva de tasas: cuánto sube la tasa por cada UMA adicional de salario
df_infonavit_tasas['delta_tasa'] = df_infonavit_tasas['tasa'].diff()
df_infonavit_tasas[['uma_min', 'uma_max', 'tasa', 'delta_tasa']].describe()

,uma_min,uma_max,tasa,delta_tasa
count,41.0000,41.0000,41.0000,40.0000
mean,4.4390,28.8049,0.0730,0.0017
std,1.3555,155.3113,0.0205,0.0002
min,0.0000,2.6000,0.0369,0.0014
25%,3.5000,3.6000,0.0559,0.0014
50%,4.5000,4.6000,0.0748,0.0019
75%,5.5000,5.6000,0.0904,0.0019
max,6.5000,999.0000,0.1045,0.0019


## Visualización

### Tablas

In [19]:
resumen_estadistico(df_wide, ['tiie_28', 'tasa_objetivo', 'fix_usd', 'inpc_general'])

,count,mean,std,min,25%,50%,75%,max
serie,,,,,,,,
tiie_28,3639.0000,7.9031,2.1994,4.2745,6.6055,7.8300,9.3308,11.5669
tasa_objetivo,3640.0000,7.6172,2.2046,4.0000,6.2500,7.5000,9.0000,11.2500
fix_usd,3639.0000,19.2834,1.4341,16.3357,18.3526,19.1953,20.1443,25.1185
inpc_general,3615.0000,117.3404,16.7984,90.3577,103.1080,113.8990,133.6810,145.8310


In [20]:
# Tabla resumen: rango de tasas Infonavit por decil de UMA
df_infonavit_tasas[['uma_min', 'uma_max', 'salario_mensual_referencia', 'tasa']].iloc[::4]

,uma_min,uma_max,salario_mensual_referencia,tasa
0,0.0000,2.6000,9272.1800,0.0369
4,2.9000,3.0000,10698.6700,0.0445
8,3.3000,3.4000,12125.1600,0.0521
12,3.7000,3.8000,13551.6500,0.0596
16,4.1000,4.2000,14978.1400,0.0672
20,4.5000,4.6000,16404.6300,0.0748
24,4.9000,5.0000,17831.1200,0.0819
28,5.3000,5.4000,19257.6100,0.0876
32,5.7000,5.8000,20684.1000,0.0932
36,6.1000,6.2000,22110.5900,0.0989


### Gráficas

In [21]:
fig = go.Figure()
for serie, nombre in [('tiie_28', 'TIIE 28 días'), ('tasa_objetivo', 'Tasa objetivo Banxico')]:
    fig.add_trace(go.Scatter(x=df_wide.index, y=df_wide[serie], name=nombre, mode='lines'))
fig.update_layout(
    title='Tasas de referencia de Banxico — histórico',
    xaxis_title='Fecha', yaxis_title='Tasa (%)', hovermode='x unified',
)
fig.show()

In [22]:
fig = px.line(df_wide.reset_index(), x='fecha', y='fix_usd',
              title='Tipo de cambio FIX (pesos por dólar)')
fig.update_layout(xaxis_title='Fecha', yaxis_title='MXN/USD')
fig.show()

In [23]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_wide.index, y=df_wide['inflacion_anual_pct'],
                          name='Inflación interanual (%)', mode='lines'))
fig.add_hline(y=3, line_dash='dot', annotation_text='Meta Banxico (3%)', line_color='green')
fig.update_layout(
    title='Inflación interanual (derivada del INPC vía Banxico, serie SP1)',
    xaxis_title='Fecha', yaxis_title='Variación % anual',
)
fig.show()

In [24]:
fig = px.bar(df_infonavit_tasas, x='salario_mensual_referencia', y='tasa',
             title='Tasa Infonavit diferenciada por nivel salarial (vigente UMA 2026)',
             labels={'salario_mensual_referencia': 'Salario mensual (MXN)', 'tasa': 'Tasa anual'})
fig.update_layout(yaxis_tickformat='.1%')
fig.show()

## Modelado de datos y evaluación de resultados

Dos modelos, uno por cada paradigma de aprendizaje, sobre las mismas fuentes reales ya cargadas en este notebook:

- **Supervisado (series de tiempo):** Prophet sobre la TIIE de Banxico, para proyectar la tasa hipotecaria de referencia a 12 meses.
- **No supervisado (clustering):** K-Means sobre las 32 entidades federativas, usando precio mediano de vivienda y variación anual — ambas variables 100% oficiales de SHF, sin depender del scraper.

Ambos se documentan con su evaluación de desempeño — backtest para Prophet, inercia y perfil de clusters para K-Means — siguiendo el mismo estándar de reproducibilidad del resto del notebook (fuentes públicas, sin credenciales).


### Modelo supervisado: Prophet — proyección de tasa hipotecaria

In [25]:
# Prophet no viene preinstalado en Colab por default
!pip install prophet --quiet

You should consider upgrading via the '/Users/luiscontreras/Documents/radar-hipotecario/.venv/bin/python3 -m pip install --upgrade pip' command.


In [26]:
from prophet import Prophet

# Reutilizamos la serie TIIE ya cargada en df_banxico (sección Data Ingestion)
df_tiie = df_banxico[df_banxico['serie'] == 'tiie_28'][['fecha', 'valor']].copy()
df_tiie = df_tiie.rename(columns={'fecha': 'ds', 'valor': 'y'}).sort_values('ds').reset_index(drop=True)
df_tiie.tail()

,ds,y
2508,2026-07-30,6.7559
2509,2026-07-31,6.7458
2510,2026-08-03,6.7559
2511,2026-08-04,6.8061
2512,2026-08-05,6.7458


In [27]:
def entrenar_prophet(df: pd.DataFrame, periods_dias: int = 365):
    """Entrena Prophet y devuelve (modelo, forecast completo)."""
    modelo = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.15,  # ver justificación de sensibilidad más abajo
    )
    modelo.fit(df)
    futuro = modelo.make_future_dataframe(periods=periods_dias, freq='D')
    return modelo, modelo.predict(futuro)

**Nota de diseño:** `changepoint_prior_scale=0.15` se eligió tras una prueba de sensibilidad (0.05 → 0.15) documentada en el desarrollo del proyecto — la mejora marginal decreciente confirmó que la fuente principal de error no es falta de flexibilidad del modelo, sino la naturaleza **discreta** de los movimientos de la TIIE (decisiones de política monetaria en fechas puntuales, no una tendencia continua). Esto se retoma en la evaluación de abajo.

### Evaluación del modelo Prophet — backtest de 365 días

In [28]:
def backtest_prophet(df: pd.DataFrame, dias_holdout: int = 365) -> dict:
    """Entrena con todo menos los últimos `dias_holdout` días, predice ese
    tramo, y compara contra los valores reales."""
    corte = df['ds'].max() - pd.Timedelta(days=dias_holdout)
    train = df[df['ds'] <= corte]
    test = df[df['ds'] > corte]

    _, forecast = entrenar_prophet(train, periods_dias=dias_holdout + 30)

    comparacion = test.merge(forecast[['ds', 'yhat']], on='ds', how='left').dropna()
    mae = (comparacion['y'] - comparacion['yhat']).abs().mean()
    rmse = ((comparacion['y'] - comparacion['yhat']) ** 2).mean() ** 0.5

    return {
        'dias_holdout': dias_holdout,
        'n_obs_comparadas': len(comparacion),
        'mae': round(mae, 4),
        'rmse': round(rmse, 4),
        'y_promedio_periodo': round(comparacion['y'].mean(), 4),
    }

metricas_backtest = backtest_prophet(df_tiie, dias_holdout=365)
metricas_backtest

21:02:37 - cmdstanpy - INFO - Chain [1] start processing
21:02:37 - cmdstanpy - INFO - Chain [1] done processing


{'dias_holdout': 365,
 'n_obs_comparadas': 251,
 'mae': np.float64(0.8339),
 'rmse': np.float64(0.8471),
 'y_promedio_periodo': np.float64(7.3275)}

In [29]:
error_relativo_pct = metricas_backtest['mae'] / metricas_backtest['y_promedio_periodo'] * 100
print(f"MAE: {metricas_backtest['mae']} puntos porcentuales")
print(f"Error relativo: {error_relativo_pct:.1f}% sobre el promedio del periodo de prueba")
print("\nInterpretación: un error relativo de esta magnitud es consistente con "
      "proyectar una serie de saltos discretos (política monetaria) con un modelo "
      "de tendencia continua — limitación esperada y documentada, no un error de ajuste.")

MAE: 0.8339 puntos porcentuales
Error relativo: 11.4% sobre el promedio del periodo de prueba

Interpretación: un error relativo de esta magnitud es consistente con proyectar una serie de saltos discretos (política monetaria) con un modelo de tendencia continua — limitación esperada y documentada, no un error de ajuste.


In [30]:
modelo_final, forecast_final = entrenar_prophet(df_tiie, periods_dias=365)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_tiie['ds'], y=df_tiie['y'], name='TIIE observada', mode='lines'))
fig.add_trace(go.Scatter(x=forecast_final['ds'], y=forecast_final['yhat'],
                          name='Proyección Prophet', mode='lines', line=dict(dash='dot')))
fig.add_trace(go.Scatter(
    x=list(forecast_final['ds']) + list(forecast_final['ds'][::-1]),
    y=list(forecast_final['yhat_upper']) + list(forecast_final['yhat_lower'][::-1]),
    fill='toself', fillcolor='rgba(0,100,80,0.1)', line=dict(width=0),
    name='Intervalo de confianza', showlegend=True,
))
fig.update_layout(title='TIIE observada vs. proyección Prophet (12 meses)',
                   xaxis_title='Fecha', yaxis_title='TIIE (%)', hovermode='x unified')
fig.show()

21:03:06 - cmdstanpy - INFO - Chain [1] start processing
21:03:06 - cmdstanpy - INFO - Chain [1] done processing


### Modelo no supervisado: K-Means — arquetipos de mercado por ciudad

In [31]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

URL_SHF_NACIONAL = f'{RAW_BASE}/config/shf_nacional.json'

In [32]:
# Feature 1: precio mediano de vivienda — dato oficial SHF, por entidad
shf_nacional = cargar_json_github(URL_SHF_NACIONAL)
df_precio_entidad = pd.DataFrame([
    {'entidad': entidad, 'precio_mediano': info['mediana']}
    for entidad, info in shf_nacional['estados'].items()
])
df_precio_entidad

,entidad,precio_mediano
0,Aguascalientes,1223582
1,Baja California,1605873
2,Baja California Sur,1897875
3,Campeche,1314572
4,Chiapas,1229650
5,Chihuahua,1344000
6,Ciudad de México,3368586
7,Coahuila,1097000
8,Colima,1144000
9,Durango,851141


In [33]:
# Feature 2: variación anual de precios — mismo archivo SHF que la Feature 1.
# Se calculó por código desde el histórico T1 2025 vs T1 2026, no a mano.
df_variacion_entidad = pd.DataFrame([
    {'entidad': entidad, 'variacion_anual_pct': info['variacion_anual_pct']}
    for entidad, info in shf_nacional['estados'].items()
])
df_variacion_entidad

,entidad,variacion_anual_pct
0,Aguascalientes,11.6600
1,Baja California,11.1200
2,Baja California Sur,11.1100
3,Campeche,7.4600
4,Chiapas,8.3800
5,Chihuahua,10.1800
6,Ciudad de México,4.4700
7,Coahuila,8.4300
8,Colima,8.4400
9,Durango,4.8900


In [34]:
df_mercado = df_precio_entidad.merge(df_variacion_entidad, on='entidad', how='inner')
df_mercado

,entidad,precio_mediano,variacion_anual_pct
0,Aguascalientes,1223582,11.6600
1,Baja California,1605873,11.1200
2,Baja California Sur,1897875,11.1100
3,Campeche,1314572,7.4600
4,Chiapas,1229650,8.3800
5,Chihuahua,1344000,10.1800
6,Ciudad de México,3368586,4.4700
7,Coahuila,1097000,8.4300
8,Colima,1144000,8.4400
9,Durango,851141,4.8900


In [35]:
FEATURES_KMEANS = ['precio_mediano', 'variacion_anual_pct']

# La regla de n_entidades // 3 (usada en el M3 de Valora AI) con 32 entidades da 10
# grupos, varios de una sola entidad — puro ruido, no información útil. Se probó y
# se descartó. k=5 fijo sí produce agrupaciones con sentido de mercado.
k = 5

X = StandardScaler().fit_transform(df_mercado[FEATURES_KMEANS])
modelo_kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df_mercado['cluster'] = modelo_kmeans.fit_predict(X)

print(f'k = {k} | inercia = {modelo_kmeans.inertia_:.3f}')
df_mercado

k = 5 | inercia = 9.700


,entidad,precio_mediano,variacion_anual_pct,cluster
0,Aguascalientes,1223582,11.6600,4
1,Baja California,1605873,11.1200,1
2,Baja California Sur,1897875,11.1100,1
3,Campeche,1314572,7.4600,0
4,Chiapas,1229650,8.3800,0
5,Chihuahua,1344000,10.1800,4
6,Ciudad de México,3368586,4.4700,2
7,Coahuila,1097000,8.4300,0
8,Colima,1144000,8.4400,0
9,Durango,851141,4.8900,3


### Evaluación del modelo K-Means — perfil e interpretación de clusters

In [36]:
perfil_clusters = df_mercado.groupby('cluster')[FEATURES_KMEANS].mean()
perfil_clusters['entidades'] = df_mercado.groupby('cluster')['entidad'].apply(list)
perfil_clusters['n_entidades'] = df_mercado.groupby('cluster')['entidad'].count()
perfil_clusters = perfil_clusters.reset_index()
perfil_clusters

,cluster,precio_mediano,variacion_anual_pct,entidades,n_entidades
0,0,1329268.0000,8.2225,"[Campeche, Chiapas, Coahuila, Colima, Guanajua...",12
1,1,1682143.0000,10.7720,"[Baja California, Baja California Sur, Morelos...",5
2,2,3368586.0000,4.4700,[Ciudad de México],1
3,3,1052366.6667,5.7200,"[Durango, México, Tabasco, Tlaxcala, Veracruz,...",6
4,4,1218629.6250,11.3125,"[Aguascalientes, Chihuahua, Jalisco, Michoacán...",8


In [37]:
def nombrar_arquetipo(precio_alto: bool, variacion_alta: bool) -> str:
    if precio_alto and variacion_alta:
        return 'Premium en expansión'
    if precio_alto and not variacion_alta:
        return 'Premium consolidado'
    if not precio_alto and variacion_alta:
        return 'Emergente'
    return 'Estable / rezagado'

precio_mediana_clusters = perfil_clusters['precio_mediano'].median()
variacion_mediana_clusters = perfil_clusters['variacion_anual_pct'].median()

for _, fila in perfil_clusters.iterrows():
    nombre = nombrar_arquetipo(
        fila['precio_mediano'] >= precio_mediana_clusters,
        fila['variacion_anual_pct'] >= variacion_mediana_clusters,
    )
    print(f"Cluster {int(fila['cluster'])} — {nombre}: {fila['entidades']}")

# Nota: con k=5 pero solo 4 combinaciones posibles de nombre (precio alto/bajo x
# variación alta/baja), es normal que dos clusters distintos terminen con el mismo
# nombre de arquetipo — no es un error, es una consecuencia de tener más grupos
# que categorías posibles al nombrarlos así.

Cluster 0 — Premium en expansión: ['Campeche', 'Chiapas', 'Coahuila', 'Colima', 'Guanajuato', 'Guerrero', 'Hidalgo', 'Nuevo León', 'Oaxaca', 'Puebla', 'Querétaro', 'San Luis Potosí']
Cluster 1 — Premium en expansión: ['Baja California', 'Baja California Sur', 'Morelos', 'Nayarit', 'Yucatán']
Cluster 2 — Premium consolidado: ['Ciudad de México']
Cluster 3 — Estable / rezagado: ['Durango', 'México', 'Tabasco', 'Tlaxcala', 'Veracruz', 'Zacatecas']
Cluster 4 — Emergente: ['Aguascalientes', 'Chihuahua', 'Jalisco', 'Michoacán', 'Quintana Roo', 'Sinaloa', 'Sonora', 'Tamaulipas']


In [38]:
fig = px.scatter(
    df_mercado, x='precio_mediano', y='variacion_anual_pct', color='cluster',
    text='entidad',
    title='Arquetipos de mercado: precio mediano de vivienda vs. variación anual',
    labels={'precio_mediano': 'Precio mediano de vivienda (MXN)', 'variacion_anual_pct': 'Variación anual (%)'},
)
fig.update_traces(textposition='top center')
fig.show()

**Nota de limitación:** el clustering opera sobre 32 puntos (las entidades federativas). A diferencia de una versión anterior de este análisis, las dos variables (precio mediano y variación anual) vienen ahora del mismo archivo oficial de SHF — el K-Means ya no depende del scraper de oferta inmobiliaria para ninguna de las dos.


## Conclusiones finales

**Hallazgos del EDA:**
- La TIIE y la tasa objetivo se mueven en escalones discretos (decisiones de política monetaria), no en tendencia continua — hallazgo que explica el desempeño del modelo Prophet más abajo.
- La curva de tasas Infonavit es progresiva y aproximadamente lineal entre 2.6 y 6.6 UMA de salario.

**Hallazgos de la modelación:**
- **Prophet** proyecta razonablemente la tendencia de la TIIE, pero con un MAE de backtest que refleja la naturaleza discreta de la serie — limitación esperada y documentada, no un error de ajuste de hiperparámetros (se probó sensibilidad de `changepoint_prior_scale` con ganancia marginal decreciente).
- **K-Means** separa las 32 entidades en 5 clusters con sentido de mercado: Ciudad de México queda sola por tener un precio muy por encima del resto (no es un error del algoritmo, es un caso real aparte); el resto se agrupa en bloques que sí coinciden con lo esperado — estados con mercados caros y en expansión (Baja California, Baja California Sur, Morelos, Nayarit, Yucatán), estados emergentes con alta variación anual (Jalisco, Aguascalientes, Chihuahua, Quintana Roo, Sinaloa, Sonora, entre otros), y estados más estables o rezagados (Durango, México, Tabasco, Tlaxcala, Veracruz, Zacatecas). Ambas variables del clustering son 100% oficiales de SHF.